### Introduction: CFPB Consumer Complaint Database

#### Purpose and Scope

The **Consumer Financial Protection Bureau (CFPB)** maintains a public database of complaints submitted by consumers regarding financial products and services. Its primary purpose is to provide transparency into the marketplace, allow regulators and researchers to detect systemic issues, and ensure financial institutions respond to consumer grievances in a timely manner.

Complaints are published after companies respond or after 15 days, provided a commercial relationship is confirmed.

#### Complaint Categories and Available Fields

The dataset contains structured tabular metadata combined with unstructured text fields. Key metadata fields include:

* **Product & Sub-product:** Broad and specific financial product classifications (e.g., *Credit reporting*, *Mortgage*, *Debt collection*, *Student loans*).

* **Issue & Sub-issue:** Categorical labels detailing the specific problem (e.g., *Incorrect information on your report*, *Trouble during payment process*).

* **Company Information:** The targeted financial institution and whether their response was timely.

* **Geographic & Temporal Data:** `State`, `ZIP code`, and `Date received`.

* **Consumer Consent & Relational Status:** Flags indicating whether consent was provided to publish the written narrative publicly.

#### The Role of Complaint Narratives in NLP

The `Consumer complaint narrative` field represents raw, unstructured text written by consumers describing their experiences in their own words. Because structured categories (`Product`/`Issue`) are selected via fixed dropdowns, they don't always capture nuances or emerging pain points.

Using Natural Language Processing (NLP) on these narratives allows us to:

* Extract topics and semantic clusters beyond pre-defined categories.
* Build automated text classification models to route or tag complaints.
* Identify recurring patterns, systemic fraud, or sudden policy shifts in real time.
* Perform sentiment and emotion analysis to gauge consumer frustration levels and top     keywords they used from their narrative complaints.


#### Project Focus: 2025 Complaints

While the full CFPB dataset dates back to 2011 and contains millions of records, this notebook focuses specifically on **complaints filed during the calendar year 2025** to perform sentiment and emotion analysis to gauge consumer frustration levels and top     keywords they used from their narrative complaints..

#### Resources & Documentation

* **CFPB Search Interface & Data Source:** [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/)
* **Technical API & Field Definitions:** [CFPB API & Data Reference Dictionary](https://cfpb.github.io/api/ccdb/api.html)

#### Load all requirements

In [1]:
#check python version
!python3 --version

Python 3.12.13


In [1]:
# !pip install pandas numpy
# !pip install -U spacy
# !pip install -U streamlit
# !pip install -U plotly
# !pip install -U matplotlib
# !pip install -U vaderSentiment
# !pip install -U textblob
# !pip install -U wordcloud
# !pip install -U scikit-learn
# !pip install -U sentence-transformers
# !pip install -U bertopic
# !pip install -U transformers
# !pip install -U torch
# !pip install vaderSentiment
# !pip install wordcloud tqdm

In [2]:
import pandas as pd
import argparse
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from spacy.matcher import PhraseMatcher
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from collections import Counter

Mounted at /content/drive


#### Loading 2025 complaints dataset

In [7]:
def load_complaints_data(file_name, is_colab=False):
    """Loads the complaints data from either a GitHub path or Google Drive.

    Args:
        file_name (str): The relative path to the CSV file (e.g., "../../data/complaints_in_just_2025.csv").
        is_colab (bool): If True, construct path for Google Drive. Defaults to False.

    Returns:
        pandas.DataFrame: The loaded and cleaned DataFrame.
    """
    if is_colab:
        from google.colab import drive
        import os
        # Mount Google Drive
        drive.mount('/content/drive')

        # Assuming your data folder is directly under MyDrive
        gdrive_base_path = '/content/drive/MyDrive/'
        # Extract just the filename from the original file_name
        base_file_name = os.path.basename(file_name)
        full_path = os.path.join(gdrive_base_path, base_file_name)
    else:
        full_path = file_name

    df = pd.read_csv(full_path, low_memory=False).dropna(subset=['Consumer complaint narrative'])
    return df

# Load the data, setting is_colab=True since we are in a Colab environment
df = load_complaints_data("../../data/complaints_in_just_2025.csv", is_colab=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df.shape

In [ ]:
df.head(5)

In [ ]:
df.groupby('Product').size().reset_index(name='count').sort_values(
    by='count', ascending=False)

In [ ]:
df.groupby([
    'Product',
    'Sub-product'
]).size().reset_index(name='count').sort_values(by='count', ascending=False)

# Count complaints for each product and sub-product combination.
# Calculate each combination's percentage of all complaints, sort by count, and keep the top 10.
product_counts = (
    df.groupby(['Product', 'Sub-product'])
      .size()
      .reset_index(name='count')
)
product_counts['percentage'] = (
    product_counts['count'] / product_counts['count'].sum() * 100
).round(2)
product_counts.sort_values(by='count', ascending=False).head(10)

In [ ]:
# The 2025 dataset has 5,443,005 rows and 16 columns
# The full dataset has 15,896,496 observations
df.shape

In [ ]:
# List the raw data types of all columns of the Pandas data frame
df.dtypes

In [ ]:
# Convert date columns to datetime and normalize timestamps to midnight (YYYY-MM-DD).
# pd.to_datetime handles both string and already-converted datetime values.
df['Date received'] = pd.to_datetime(
    df['Date received'], errors='coerce').dt.normalize()

df['Date sent to company'] = pd.to_datetime(
    df['Date sent to company'], errors='coerce').dt.normalize()

In [ ]:
# Look at a few obs to make sure we have correctly formatformatted date and time.
df.head()

In [ ]:
# Group complaints by year received and count observations
df.groupby(df['Date received'].dt.year).size()

In [ ]:
# Group complaints by product category and count observations
df.groupby(df['Product']).size()

In [ ]:
# Group complaints by issue category and count observations

df.groupby(df['Issue']).size()

In [ ]:
# Group by the company public response and count obsevations

df.groupby(df['Company public response']).size()

In [ ]:
# Group by company response to consumer and count obsevations

df.groupby(df['Company response to consumer']).size()

In [ ]:
# Group by timely response and Company response to consumer to consumer complaints and count obsevations

df.groupby(['Timely response?','Company response to consumer']).size()

In [ ]:
# Count up the number of unique companies having consumer complaints

df['Company'].nunique()

In [ ]:
# Find the top 10 companies with the most consumer complaints
df['Company'].value_counts().head(10)

### Summary of the data wrangling

**Dataset loading and validation:** Import the required Python libraries and loads the 2025 CFPB complaint file into a pandas DataFrame. Rows without a `Consumer complaint narrative` are removed.

### Key insights

**Credit reporting dominates the dataset.** Credit reporting or other personal consumer reporting complaints account for the largest product category and substantially exceed the other categories. This concentration should be considered when interpreting NLP results because overall themes may primarily reflect credit-reporting experiences.

**Debt collection and money-transfer complaints are also prominent.** These categories form important secondary groups and may reveal distinct issues involving collection practices, payment transfers, and account access.

**Complaint narratives are a valuable source of unstructured information.** Product and issue fields provide structured labels, while the narratives provide consumers' descriptions of events, impacts, and desired resolutions. This supports text classification, topic discovery, and sentiment or outcome analysis.

**Company-response fields enable outcome analysis.** Comparing `Company response to consumer` with `Timely response?` can show whether timely handling is associated with different resolution outcomes. These fields describe reported responses, not necessarily whether the consumer considered the issue fully resolved.

**Company counts may reflect market size as well as complaint frequency.** Companies with the most complaints may serve more customers or offer more products, so complaint volume should not be interpreted as a complaint rate without an appropriate exposure denominator.

**Privacy masking affects text modeling.** CFPB narratives may contain redacted names, dates, account numbers, and other identifying information. These masks should be handled during preprocessing so they do not dominate the vocabulary or create misleading NLP features.

In [ ]:
df_2025 = df.copy()
df_2025.rename(
    columns={'Consumer complaint narrative': 'narrative'},
    inplace=True
)

## Data preparation
The CFPB database is massive, but not all entries contain text because consumers must explicitly opt-in to share their narratives.

Here is a step-by-step pipeline to clean, preprocess, and structure this data for Natural Language Processing (NLP).

* Filter out rows where Consumer complaint narrative is null.

* The CFPB heavily sanitizes data for privacy, replacing names, account numbers, and dates with blocks of XXXX. They hold no sentiment
value and can confuse word embeddings. Therefore, remove them or replace them with a single space.

In [ ]:
#load modules
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
df = df.dropna(subset=['Consumer complaint narrative'])
# Rename for ease
df['narrative'] = df['Consumer complaint narrative']

df.head()

## Text Preprocessing Pipeline

Consumer complaint narratives are unstructured text and may contain inconsistent capitalization, punctuation, web artifacts, privacy masks, and boilerplate language. Before applying machine-learning models such as TF-IDF with logistic regression or more advanced deep-learning methods, the narratives must be normalized to reduce noise and create a consistent vocabulary.

The preprocessing workflow includes the following steps:

- **Normalize privacy masks:** Standardize CFPB redactions such as `XXXX`, masked dates, and masked dollar amounts so that they are treated consistently rather than as unrelated tokens.

- **Convert text to lowercase:** Make words such as `Credit` and `credit` equivalent, reducing unnecessary vocabulary duplication.

- **Remove web artifacts:** Remove URLs, HTML entities, duplicate whitespace, and other formatting artifacts that do not contribute meaningful information.

- **Remove punctuation and special characters:** Reduce noise while preserving information that may be useful for interpretation.

- **Tokenize the narratives:** Split each narrative into individual words or tokens for analysis.

- **Remove stop words selectively:** Remove frequent words with limited analytical value, but retain negation terms such as `not` and `no` because they can change the meaning of a complaint.

- **Lemmatize words:** Reduce related word forms to a common base form; for example, `charged`, `charging`, and `charges` can be represented by `charge`.

- **Filter uninformative records:** Remove empty, whitespace-only, duplicate, or very short narratives that are unlikely to provide sufficient information for modeling.

This sequence produces cleaner text while preserving important complaint meaning and resolution-related language for downstream NLP analysis.

In [ ]:
def normalize_masks(text):
    """Replace CFPB privacy redactions with meaningful standard tokens."""
    # Replace masked dates, such as XX/XXX/XXXXX, with a consistent date token.
    text = re.sub(r'\b[xX]{2}/[xX]{3}/[xX]{4\b', '[MASKED_DATE]', text)
    # Replace masked dollar amounts or numeric values with an amount token.
    text = re.sub(r'\$[xX,]+', '[MASKED_AMOUNT]', text)
    # Replace remaining sequences of two or more x/X characters with a text token.
    text = re.sub(r'[xX]{2,}', '[MASKED_TEXT]', text)
    # Apply the standardized masking function to the complaint narratives.
    text = re.sub(r'X{2,}', '[REDACTED]', text)
    return text

# Create a cleaned text column while preserving the original complaint narrative.
df_2025['cleaned_narrative'] = df_2025['narrative'].apply(normalize_masks)

In [ ]:
# Filter out short, non-meaningful descriptions (under 20 words)
df_2025 = df_2025[df_2025['cleaned_narrative'].apply(lambda x: len(str(x).split()) > 20)]

In [ ]:
import html

def clean_web_artifacts(text):
    text = html.unescape(text) # Converts &amp; to &
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Removes URLs
    text = re.sub(r'\s+', ' ', text).strip() # Removes duplicate whitespace/newlines
    return text

In [ ]:
# Remove web and formatting artifacts from the cleaned complaint narratives.
df_2025['cleaned_narrative'] = df_2025['cleaned_narrative'].apply(clean_web_artifacts)

#### Grouping shifting categorical targets to ensure text consistency

In [ ]:

product_mapping = {
    'Credit card': 'Credit Card/Prepaid',
    'Prepaid card': 'Credit Card/Prepaid',
    'Credit card or prepaid card': 'Credit Card/Prepaid',
}
df_2025['Product_Standardized'] = df_2025['Product'].replace(product_mapping)

#### Keep only rows where a commercial relationship was verified and sent to the company

In [ ]:
# Keep only rows where a commercial relationship was verified and sent to the company
df_2025 = df_2025[df_2025['Submitted via'] != 'Referral']
df_2025 = df_2025.dropna(subset=['Company response to consumer'])

#### Prune Document Length Extremes
Extremely long documents dilate the computational matrix size (especially for deep learning embedding vectors) and dilute the actual "sentiment" signal with thousands of words of standard chronological noise.

In [ ]:
# Filter by token-count percentiles
df_2025['word_count'] = df_2025['cleaned_narrative'].apply(lambda x: len(x.split()))
min_thresh = df_2025['word_count'].quantile(0.05)
max_thresh = df_2025['word_count'].quantile(0.95)

df_filtered = df_2025[(df_2025['word_count'] >= min_thresh) & (df_2025['word_count'] <= max_thresh)]

In [ ]:
df_filtered.shape

In [ ]:
# Map responses to Binary Sentiment/Outcome (1 = Relief/Positive resolution, 0 = No relief)
relief_responses = ['Closed with monetary relief', 'Closed with non-monetary relief']
df_filtered['response_binary'] = df_filtered['Company response to consumer'].apply(lambda x: 1 if x in relief_responses else 0)

In [ ]:
df_filtered.head()

In [ ]:
# save the filtered DataFrame to a new CSV file
#This is the same as df_filtered csv file
df_filtered.to_csv('../../data/complaints_2025_filtered.csv', index=False)


#### SpaCy

In [ ]:
df=df_filtered.sample(10000)

In [ ]:
df.head()

In [ ]:
!python3 -m spacy download en_core_web_sm
!python3 -m spacy download en_core_web_md
!python3 -m spacy download en_core_web_lg

## Sentiment Analysis: VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based sentiment analysis tool designed for short, informal text. It assigns a compound sentiment score ranging from -1 to 1, where values closer to -1 indicate more negative sentiment and values closer to 1 indicate more positive sentiment. The score is calculated from the cleaned complaint narrative and is used to summarize the emotional tone of consumers' descriptions.

Because CFPB complaint narratives are often detailed, formal, and focused on specific financial problems, VADER scores should be interpreted as an indicator of expressed sentiment rather than a direct measure of complaint severity, customer satisfaction, or resolution success.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_score(text):
    score = analyzer.polarity_scores(text)
    return score["compound"]

In [ ]:
df["sentiment_vader"] = df["cleaned_narrative"].apply(vader_score)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['sentiment_vader'], bins=30, kde=True)
plt.title('Distribution of Sentiment Scores')
plt.xlabel('Sentiment Score')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
# Compare average VADER sentiment across the 10 most common complaint issues.
top_issues = df['Issue'].value_counts().head(10).index
issue_sentiment = (
    df[df['Issue'].isin(top_issues)]
      .groupby('Issue', as_index=False)['sentiment_vader']
      .mean()
      .sort_values('sentiment_vader', ascending=False)
)

plt.figure(figsize=(12, 7))
sns.barplot(
    data=issue_sentiment,
    x='sentiment_vader',
    y='Issue',
    hue='Issue',
    palette='tab10',
    legend=False
)
plt.title('Average VADER Sentiment by Top Complaint Issue')
plt.xlabel('Average VADER Compound Score')
plt.ylabel('Complaint Issue')
plt.axvline(0, color='black', linewidth=1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Interpreting the positive VADER scores

A large number of positive VADER scores does not necessarily mean that consumers were satisfied with their financial products or that their complaints were resolved positively. Several factors can explain this pattern:

- **Complaint narratives contain mixed language.** Consumers may describe a negative experience while also using positive words such as `help`, `thank`, `appreciate`, `correct`, or `resolved` when discussing a company representative, a requested outcome, or a partial improvement.

- **VADER is lexicon-based.** VADER assigns sentiment values to individual words and phrases. It may identify positive terms without fully understanding the broader financial context or the complaint's main issue.

- **Narratives often include procedural language.** Words such as `account`, `payment`, `credit`, `service`, and `information` can appear in contexts that are not emotionally positive, but the overall wording may still produce a slightly positive compound score.

- **The cleaned text changes the original writing.** Lowercasing, removing punctuation, filtering tokens, and replacing redacted information can remove emphasis, negation cues, or other context that affects sentiment scoring.

- **A positive score is not the same as a positive outcome.** VADER measures the tone expressed in the narrative; it does not measure whether the consumer received monetary relief, non-monetary relief, or a satisfactory resolution.

Therefore, VADER scores should be interpreted together with the original narrative, product and issue categories, and the company-response fields. They are most useful for comparing broad language patterns across groups rather than classifying individual complaints as resolved or unresolved.

# Financial Severity Score

In [ ]:

nlp = spacy.load("en_core_web_sm")

# Define categories and keyword lists
category_lexicon = {
    # Severity Levels
    "critical": [
        "foreclosure", "eviction", "bankruptcy", "homeless", "sheriff sale",
        "lease termination", "padlock", "shelter", "lawsuit", "garnishment",
        "subpoena", "attorney general", "summons", "court order", "judgment",
        "wage attachment", "poverty", "destitute", "hardship", "suicidal"
    ],
    "high": [
        "fraud", "identity theft", "stolen funds", "scam", "unauthorized transaction",
        "account takeover", "phishing", "impersonation", "wire fraud", "forgery",
        "closed account", "repossession", "collections", "frozen account",
        "seized funds", "account lock", "charge off", "predatory lending", "harassment"
    ],
    "medium": [
        "incorrect reporting", "credit score", "dispute", "fcra", "unverified debt",
        "mixed file", "late fee", "overdraft fee", "hidden charge", "unauthorized fee",
        "interest rate increase", "billing error", "escrow shortage", "double charged"
    ],

    # Domain / Action Types
    "legal_risk": [
        "lawsuit", "attorney", "lawyer", "court", "subpoena", "legal action",
        "suing", "fcra violation", "fdcpa", "tcpa", "regulatory complaint"
    ],
    "financial_loss": [
        "stolen funds", "overdraft fee", "unauthorized charge", "drained account",
        "wire fraud", "double charged", "seized funds", "unauthorized fee"
    ],
    "credit_damage": [
        "credit score dropped", "incorrect reporting", "derogatory mark",
        "late payment reported", "identity theft", "mixed file", "unverified debt"
    ]
}


# Severity Score Function

In [ ]:
# Initialize matcher
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

# Register each category into the matcher
for cat_name, terms in category_lexicon.items():
    patterns = [nlp.make_doc(text) for text in terms]
    matcher.add(cat_name, patterns)

In [ ]:
def score_multi_category(doc):
    """
    Evaluates a spaCy Doc against registered categories and returns
    counts and calculated severity metrics.
    """
    matches = matcher(doc)

    # Track term hits per category
    cat_counts = {cat: 0 for cat in category_lexicon.keys()}

    for match_id, start, end in matches:
        string_id = nlp.vocab.strings[match_id]  # Category name
        cat_counts[string_id] += 1

    # Calculate Weighted Composite Severity Score
    # Critical = 5 pts | High = 3 pts | Medium = 1 pt
    composite_severity = (
        (cat_counts['critical'] * 5) +
        (cat_counts['high'] * 3) +
        (cat_counts['medium'] * 1)
    )

    # Return dictionary of all scores
    return {
        'count_critical': cat_counts['critical'],
        'count_high': cat_counts['high'],
        'count_medium': cat_counts['medium'],
        'count_legal_risk': cat_counts['legal_risk'],
        'count_financial_loss': cat_counts['financial_loss'],
        'count_credit_damage': cat_counts['credit_damage'],
        'composite_severity_score': composite_severity
    }

# Batch process narratives
# Ensure 'cleaned_narrative' column contains only strings, converting any NaN to empty strings
docs = list(nlp.pipe(df['cleaned_narrative'].fillna(''), disable=["ner", "parser"]))
scores_list = [score_multi_category(doc) for doc in docs]

# Convert results into a DataFrame and combine with main data
score_df = pd.DataFrame(scores_list)
df_scored = pd.concat([df.reset_index(drop=True), score_df], axis=1)

In [ ]:
#df_scored.head()

In [ ]:
# Top Companies by Severity
import plotly.express as px
company_score = (
    df_scored
    .groupby("Company")
    ["composite_severity_score"]
    .mean()
    .reset_index()
    .sort_values(
    "composite_severity_score",
    ascending=False
    )
    .head(20)
    )

fig = px.bar(
    company_score,
    title="Top 20 Companies by Average Composite Severity Score",
    x="composite_severity_score",
    y="Company",
    orientation="h"
    )

fig.show()

### Plot Insight: top 20 companies by average composite severity score
The bar chart ranks the top 20 companies by average composite severity score, showing which firms receive the most severe complaint language on average. A higher score indicates that a company's complaints contain more critical, high-risk, or financially harmful terms, so the chart helps identify companies with the most severe consumer issues in the sampled 2025 complaints.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
cat_cols = ['count_critical', 'count_high', 'count_medium', 'count_legal_risk', 'count_financial_loss', 'count_credit_damage']

sns.heatmap(df_scored[cat_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Multi-Category Keyword Correlation Heatmap')
plt.show()

In [ ]:
# Calculate total mentions for each category across the dataset
severity_totals = pd.DataFrame({
    'Severity Level': ['Critical', 'High', 'Medium'],
    'Total Mentions': [
        df_scored['count_critical'].sum(),
        df_scored['count_high'].sum(),
        df_scored['count_medium'].sum()
    ]
})

# Plot Total Mentions Bar Chart
plt.figure(figsize=(8, 5))
ax = sns.barplot(
    data=severity_totals,
    x='Severity Level',
    y='Total Mentions',
    palette=['#d9534f', '#f0ad4e', '#5bc0de']  # Red, Orange, Blue
)

for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height()):,}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom',
        fontsize=11, fontweight='bold',
        xytext=(0, 5), textcoords='offset points'
    )

plt.title('Total Keyword Frequency Across Severity Categories', fontsize=14, pad=15)
plt.xlabel('Severity Level', fontsize=12)
plt.ylabel('Total Keyword Occurrences', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:

from collections import Counter

# Additional stop words specific to CFPB
custom_stopwords = {
    "bank","company","account","credit","loan","card",
    "consumer","told","said","would","could","also",
    "one","get","got","even","back","time","called",
    "call","email","letter","know","received","receive"
}

def extract_keywords(text):
    doc = nlp(str(text).lower())

    words = []

    for token in doc:

        if (
            token.is_stop
            or token.is_punct
            or token.is_space
            or token.like_num
        ):
            continue

        if token.lemma_ in custom_stopwords:
            continue

        if token.pos_ in ["NOUN", "PROPN", "ADJ"]:

            if len(token.lemma_) > 2:
                words.append(token.lemma_)

    return words

In [ ]:
issue_keywords = {}

for issue, group in df_scored.groupby("Issue"):

    counter = Counter()

    for text in group["cleaned_narrative"]:

        counter.update(extract_keywords(text))

    issue_keywords[issue] = counter

for issue, counter in issue_keywords.items():

    print("\n", issue)
    print(counter.most_common(20))

In [ ]:
from wordcloud import WordCloud

for issue, counter in issue_keywords.items():

    wc = WordCloud(
        width=1000,
        height=600,
        background_color="white",
        colormap="tab20"
    )

    wc.generate_from_frequencies(dict(counter))

    plt.figure(figsize=(12,6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(issue, fontsize=18)
    plt.show()

In [ ]:
rows = []

for issue, counter in issue_keywords.items():

    for word, freq in counter.most_common(20):

        rows.append({
            "Issue": issue,
            "Keyword": word,
            "Frequency": freq
        })

top20 = pd.DataFrame(rows)

top20.to_csv("Top20_Keywords_By_Issue.csv", index=False)

top20.head()